In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hope-english/english_hope_dev.csv
/kaggle/input/hope-english/english_hope_test.csv
/kaggle/input/hope-english/english_hope_train.csv


In [2]:
import pandas as pd

train = pd.read_csv("/kaggle/input/hope-english/english_hope_train.csv", sep="\t", header=None)
dev   = pd.read_csv("/kaggle/input/hope-english/english_hope_dev.csv", sep="\t", header=None)
test  = pd.read_csv("/kaggle/input/hope-english/english_hope_test.csv", sep="\t", header=None)

train.columns = ["raw"]
dev.columns = ["raw"]
test.columns = ["raw"]


In [3]:
def clean_split(df):
    df = df.copy()

    # split by ";" into max 3 parts
    parts = df["raw"].str.split(";", expand=True)

    # parts[0] = text  
    # parts[1] = label
    df["text"] = parts[0]
    df["label"] = parts[1]

    # drop rows without labels
    df = df.dropna(subset=["label"])
    
    # keep only text + label
    df = df[["text", "label"]]
    return df

train = clean_split(train)
dev = clean_split(dev)
test = clean_split(test)

print(train.head())
print(train.shape)


                                                text            label
0  these tiktoks radiate gay chaotic energy and i...  Non_hope_speech
1  @Champions Again He got killed for using false...  Non_hope_speech
2               It's not that all lives don't matter  Non_hope_speech
3  Is it really that difficult to understand? Bla...  Non_hope_speech
4  Whenever we say black isn't that racists?  Why...  Non_hope_speech
(22762, 2)


In [4]:
import re

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+|www\S+", "", t)       # remove URLs
    t = re.sub(r"@\w+", "", t)                # remove usernames
    t = re.sub(r"\s+", " ", t).strip()        # remove extra spaces
    return t

train["text"] = train["text"].apply(clean_text)
dev["text"] = dev["text"].apply(clean_text)
test["text"] = test["text"].apply(clean_text)


In [5]:
label_map = {
    "Hope_speech": 1,
    "Non_hope_speech": 0
}

train["label"] = train["label"].map(label_map)
dev["label"]   = dev["label"].map(label_map)
test["label"]  = test["label"].map(label_map)


In [6]:
train = train.dropna(subset=["label"])
dev   = dev.dropna(subset=["label"])
test  = test.dropna(subset=["label"])


In [7]:
print(train["label"].value_counts())


label
0.0    20700
1.0     1945
Name: count, dtype: int64


In [8]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [9]:
from sentence_transformers import SentenceTransformer

# Load the T5 sentence embedding model
# You can use 'sentence-t5-base' or 'sentence-t5-small' for faster computation
model = SentenceTransformer('sentence-t5-base')

# Suppose your text data is in a pandas DataFrame called train, dev, test
train_texts = train["text"].tolist()
dev_texts   = dev["text"].tolist()
test_texts  = test["text"].tolist()

# Encode the texts into sentence embeddings
# Output: numpy array of shape (num_samples, embedding_dim)
train_embeddings = model.encode(train_texts, batch_size=32, show_progress_bar=True)
dev_embeddings   = model.encode(dev_texts, batch_size=32, show_progress_bar=True)
test_embeddings  = model.encode(test_texts, batch_size=32, show_progress_bar=True)

print("Train embeddings shape:", train_embeddings.shape)
print("Dev embeddings shape:", dev_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

2025-12-02 16:59:12.530629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764694752.950965      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764694753.069422      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

2_Dense/rust_model.ot:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/708 [00:00<?, ?it/s]

Batches:   0%|          | 0/89 [00:00<?, ?it/s]

Batches:   0%|          | 0/89 [00:00<?, ?it/s]

Train embeddings shape: (22645, 768)
Dev embeddings shape: (2830, 768)
Test embeddings shape: (2828, 768)


In [10]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert embeddings to PyTorch tensors
X_train = torch.tensor(train_embeddings, dtype=torch.float32)
X_dev   = torch.tensor(dev_embeddings, dtype=torch.float32)
X_test  = torch.tensor(test_embeddings, dtype=torch.float32)

# Convert labels to tensors
y_train = torch.tensor(train["label"].values, dtype=torch.long)
y_dev   = torch.tensor(dev["label"].values, dtype=torch.long)
y_test  = torch.tensor(test["label"].values, dtype=torch.long)

# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
dev_dataset   = TensorDataset(X_dev, y_dev)
test_dataset  = TensorDataset(X_test, y_test)

# Create DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train loader batches:", len(train_loader))
print("Dev loader batches:", len(dev_loader))
print("Test loader batches:", len(test_loader))


Train loader batches: 708
Dev loader batches: 89
Test loader batches: 89


In [11]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class HopeSpeechCNN(nn.Module):
#     def __init__(self, embedding_dim=768, dropout_rate=0.25, num_classes=2):
#         super(HopeSpeechCNN, self).__init__()
        
#         # Fully connected layer before CNN
#         self.fc1 = nn.Linear(embedding_dim, 1536)
#         self.dropout = nn.Dropout(dropout_rate)
        
#         # 3 Conv1D layers
#         self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=5)
#         self.conv2 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=5)
#         self.conv3 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=5)
        
#         # MaxPooling layers (pool size 4)
#         self.pool = nn.MaxPool1d(kernel_size=4)
        
#         # Final Dense layer
#         # Compute flattened size after conv+pool:
#         # input_len = 1536
#         # After conv1 (kernel 5): 1536-5+1 = 1532
#         # After pool1 (4): 1532//4 = 383
#         # After conv2 (5): 383-5+1=379
#         # After pool2 (4): 379//4=94
#         # After conv3 (5): 94-5+1=90
#         # After pool3 (4): 90//4=22
#         # Flatten size = 22*64=1408
#         self.fc_out = nn.Linear(22*64, num_classes)
        
#     def forward(self, x):
#         # x shape: (batch_size, embedding_dim)
#         x = self.fc1(x)           # Fully connected
#         x = F.relu(x)
#         x = self.dropout(x)
        
#         # Prepare for Conv1D: (batch, channels, seq_len)
#         x = x.unsqueeze(1)        # Add channel dimension: (batch, 1, 1536)
        
#         # Conv + ReLU + Pool layers
#         x = self.conv1(x)
#         x = F.relu(x)
#         x = self.pool(x)
        
#         x = self.conv2(x)
#         x = F.relu(x)
#         x = self.pool(x)
        
#         x = self.conv3(x)
#         x = F.relu(x)
#         x = self.pool(x)
        
#         # Flatten and final dense
#         x = x.view(x.size(0), -1)
#         x = self.fc_out(x)
#         return F.softmax(x, dim=1)


In [15]:
class HopeSpeechCNN(nn.Module):
    def __init__(self, embedding_dim=768, dropout_rate=0.25, num_classes=2):
        super().__init__()
        
        self.fc1 = nn.Linear(embedding_dim, 1536)
        self.dropout = nn.Dropout(dropout_rate)

        self.conv1 = nn.Conv1d(1, 64, kernel_size=5)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=5)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=5)

        self.pool = nn.MaxPool1d(kernel_size=4)

        self.fc_out = nn.Linear(22 * 64, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = x.unsqueeze(1)

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)
        return self.fc_out(x)          # ❗ No softmax here


In [16]:
from sklearn.metrics import f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, train_loader, dev_loader, epochs=10, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        dev_acc, dev_f1 = evaluate(model, dev_loader)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {total_loss/len(train_loader):.4f}")
        print(f"Dev Accuracy: {dev_acc:.4f} | Dev Weighted F1: {dev_f1:.4f}\n")

    return model


    return model


In [19]:
def evaluate(model, loader):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())

    # compute accuracy and macro F1
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average="macro")

    return acc, f1



In [20]:
model = HopeSpeechCNN(num_classes=2)
model = train_model(model, train_loader, dev_loader, epochs=10, lr=1e-4)


Epoch 1/10 | Train Loss: 0.2852
Dev Accuracy: 0.9039 | Dev Weighted F1: 0.4748

Epoch 2/10 | Train Loss: 0.1860
Dev Accuracy: 0.9095 | Dev Weighted F1: 0.7733

Epoch 3/10 | Train Loss: 0.1777
Dev Accuracy: 0.8806 | Dev Weighted F1: 0.7482

Epoch 4/10 | Train Loss: 0.1736
Dev Accuracy: 0.9276 | Dev Weighted F1: 0.7323

Epoch 5/10 | Train Loss: 0.1721
Dev Accuracy: 0.9300 | Dev Weighted F1: 0.7889

Epoch 6/10 | Train Loss: 0.1699
Dev Accuracy: 0.9290 | Dev Weighted F1: 0.7911

Epoch 7/10 | Train Loss: 0.1681
Dev Accuracy: 0.9311 | Dev Weighted F1: 0.7939

Epoch 8/10 | Train Loss: 0.1659
Dev Accuracy: 0.9318 | Dev Weighted F1: 0.8001

Epoch 9/10 | Train Loss: 0.1628
Dev Accuracy: 0.9304 | Dev Weighted F1: 0.7987

Epoch 10/10 | Train Loss: 0.1609
Dev Accuracy: 0.9113 | Dev Weighted F1: 0.7862

